In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
ratings_with_books = pd.read_csv(
    '../data/processed/ratings_with_books.csv',
    dtype={'isbn': str}
)

books = pd.read_csv(
    '../data/processed/books_clean.csv',
    dtype={'isbn': str}
)

ratings_with_books.head()

In [ ]:

book_stats = (
    ratings_with_books
    .groupby('title')
    .agg(
        num_ratings=('rating', 'count'),
        avg_rating=('rating', 'mean')
    )
    .reset_index()
)

book_stats.head()


In [ ]:

C = ratings_with_books['rating'].mean()
m = 50

book_stats['weighted_score'] = (
    (book_stats['num_ratings'] /
     (book_stats['num_ratings'] + m)) *
    book_stats['avg_rating']
    +
    (m /
     (book_stats['num_ratings'] + m)) * C
)


In [ ]:

top_books = book_stats.sort_values(
    'weighted_score',
    ascending=False
).head(20)

top_books


In [ ]:

edition_counts = (
    ratings_with_books
    .groupby(['title', 'isbn'])
    .size()
    .reset_index(name='edition_ratings')
)

representative_editions = (
    edition_counts
    .sort_values(
        ['title', 'edition_ratings', 'isbn'],
        ascending=[True, False, True]
    )
    .drop_duplicates('title')
)

book_metadata = (
    representative_editions[['title', 'isbn']]
    .merge(
        books[['isbn', 'author', 'image_m']],
        on='isbn',
        how='left'
    )
)

popular_recommendations = top_books.merge(
    book_metadata,
    on='title',
    how='left'
)

popular_recommendations[
    ['title', 'author', 'num_ratings',
     'avg_rating', 'weighted_score', 'image_m']
].head(10)


In [ ]:

from pathlib import Path

model_dir = Path('../models')
model_dir.mkdir(parents=True, exist_ok=True)

popularity_ranking = (
    book_stats
    .sort_values('weighted_score', ascending=False)
    .merge(book_metadata, on='title', how='left')
)

popularity_ranking.to_csv(
    model_dir / 'popularity_ranking.csv',
    index=False
)

print("Popularity ranking saved!")


## Collaborative filtering

In [ ]:

user_counts = (
    ratings_with_books
    .groupby('user_id')['isbn']
    .nunique()
)

for threshold in [5, 10, 20, 50, 100, 200]:
    print(
        f"Users with >= {threshold} ratings:",
        (user_counts >= threshold).sum()
    )


In [ ]:

book_counts = (
    ratings_with_books
    .groupby('isbn')['user_id']
    .nunique()
)

for threshold in [2, 5, 10, 20, 50, 100]:
    print(
        f"Books with >= {threshold} ratings:",
        (book_counts >= threshold).sum()
    )


In [ ]:

active_users = user_counts[
    user_counts >= 10
].index

frequent_books = book_counts[
    book_counts >= 5
].index

filtered_ratings = ratings_with_books[
    ratings_with_books['user_id'].isin(active_users)
    &
    ratings_with_books['isbn'].isin(frequent_books)
].copy()

print("Filtered interactions:", len(filtered_ratings))
print("Users:", filtered_ratings['user_id'].nunique())
print("Books:", filtered_ratings['isbn'].nunique())


In [ ]:

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
import numpy as np
import pandas as pd


In [ ]:

user_interaction_counts = (
    filtered_ratings.groupby('user_id')['isbn'].nunique()
)

eligible_users = user_interaction_counts[
    user_interaction_counts >= 2
].index

model_data = filtered_ratings[
    filtered_ratings['user_id'].isin(eligible_users)
].copy()

print("Eligible users:", model_data['user_id'].nunique())
print("Interactions:", len(model_data))


In [ ]:

test_data = (
    model_data
    .groupby('user_id', group_keys=False)
    .sample(n=1, random_state=42)
)

train_data = model_data.drop(test_data.index).copy()

print("Training interactions:", len(train_data))
print("Testing interactions:", len(test_data))


In [ ]:

from scipy.sparse import csr_matrix

train_books = train_data['isbn'].unique()
train_users = train_data['user_id'].unique()

book_to_idx = {
    isbn: idx for idx, isbn in enumerate(train_books)
}

user_to_idx = {
    user: idx for idx, user in enumerate(train_users)
}

row_indices = train_data['isbn'].map(book_to_idx)
col_indices = train_data['user_id'].map(user_to_idx)

book_user_matrix = csr_matrix(
    (
        train_data['rating'].astype(float),
        (row_indices, col_indices)
    ),
    shape=(len(train_books), len(train_users))
)

print("Matrix shape:", book_user_matrix.shape)
print("Non-zero entries:", book_user_matrix.nnz)


In [ ]:

knn_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=11,
    n_jobs=1
)

knn_model.fit(book_user_matrix)

print("Collaborative filtering model trained!")


In [ ]:

isbn_to_title = (
    books.drop_duplicates('isbn')
    .set_index('isbn')['title']
    .to_dict()
)

def recommend_similar_books(isbn, n=5):
    if isbn not in book_to_idx:
        return pd.DataFrame({
            'message': ['Book not found in trained model']
        })

    book_idx = book_to_idx[isbn]

    distances, indices = knn_model.kneighbors(
        book_user_matrix[book_idx],
        n_neighbors=min(n + 1, len(train_books))
    )

    recommendations = []

    for distance, idx in zip(distances[0], indices[0]):
        recommended_isbn = train_books[idx]

        if recommended_isbn == isbn:
            continue

        recommendations.append({
            'isbn': recommended_isbn,
            'title': isbn_to_title.get(recommended_isbn, 'Unknown'),
            'similarity': round(1 - distance, 4)
        })

        if len(recommendations) == n:
            break

    return pd.DataFrame(recommendations)


In [ ]:

books[
    books['title'].str.contains(
        'The Namesake',
        case=False,
        na=False
    )
][['isbn', 'title']].head(10)


In [ ]:

recommend_similar_books('0395927218', n=5)


In [ ]:

def get_user_recommendations(user_id, n=10):
    user_history = train_data[
        (train_data['user_id'] == user_id) &
        (train_data['rating'] >= 7)
    ]

    seen_books = set(
        train_data.loc[
            train_data['user_id'] == user_id, 'isbn'
        ]
    )

    scores = {}

    for isbn in user_history['isbn']:
        if isbn not in book_to_idx:
            continue

        recs = recommend_similar_books(isbn, n=30)

        if 'isbn' not in recs.columns:
            continue

        for _, rec in recs.iterrows():
            candidate = rec['isbn']

            if candidate not in seen_books:
                scores[candidate] = (
                    scores.get(candidate, 0)
                    + rec['similarity']
                )

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return ranked[:n]


In [ ]:

hits = 0
evaluated_users = 0

for _, row in test_data.iterrows():
    user_id = row['user_id']
    actual_isbn = row['isbn']

    recommendations = get_user_recommendations(
        user_id, n=10
    )

    if not recommendations:
        continue

    evaluated_users += 1

    if actual_isbn in recommendations:
        hits += 1

hit_rate = (
    hits / evaluated_users
    if evaluated_users > 0 else 0
)

print("Evaluated users:", evaluated_users)
print(f"Hit Rate@10: {hit_rate:.4f}")


In [ ]:

isbn = '0439139597'

print("In book catalog:", isbn in set(books['isbn']))
print("In filtered data:", isbn in set(filtered_ratings['isbn']))
print("In training matrix:", isbn in book_to_idx)


In [ ]:

# Find the most frequently rated ISBN in training
most_rated_isbn = (
    train_data['isbn']
    .value_counts()
    .idxmax()
)

print("Selected book:")
print(isbn_to_title.get(most_rated_isbn))

print("\nRecommendations:")
display(
    recommend_similar_books(most_rated_isbn, n=10)
)


In [ ]:

source_idx = book_to_idx[most_rated_isbn]
source_vector = book_user_matrix[source_idx]

recs = recommend_similar_books(
    most_rated_isbn, n=10
).copy()

overlaps = []

for isbn in recs['isbn']:
    candidate_idx = book_to_idx[isbn]
    candidate_vector = book_user_matrix[candidate_idx]

    common_users = (
        source_vector.multiply(candidate_vector)
    ).count_nonzero()

    overlaps.append(common_users)

recs['common_users'] = overlaps

display(recs)


In [ ]:

def recommend_improved(isbn, n=10, min_common=5, shrinkage=20):

    if isbn not in book_to_idx:
        return pd.DataFrame()

    source_idx = book_to_idx[isbn]
    source_vector = book_user_matrix[source_idx]

    # Retrieve a larger candidate pool
    distances, indices = knn_model.kneighbors(
        source_vector,
        n_neighbors=min(200, len(train_books))
    )

    recommendations = []

    for distance, idx in zip(distances[0], indices[0]):

        candidate_isbn = train_books[idx]

        if candidate_isbn == isbn:
            continue

        candidate_vector = book_user_matrix[idx]

        common_users = source_vector.multiply(
            candidate_vector
        ).count_nonzero()

        if common_users < min_common:
            continue

        cosine_sim = 1 - distance

        adjusted_sim = cosine_sim * min(
            1, common_users / shrinkage
        )

        recommendations.append({
            'isbn': candidate_isbn,
            'title': isbn_to_title.get(
                candidate_isbn, 'Unknown'
            ),
            'cosine_similarity': round(cosine_sim, 4),
            'common_users': common_users,
            'adjusted_similarity': round(adjusted_sim, 4)
        })

    return (
        pd.DataFrame(recommendations)
        .sort_values(
            'adjusted_similarity',
            ascending=False
        )
        .head(n)
    )


In [ ]:

print("Original recommendations:")
display(
    recommend_similar_books(
        most_rated_isbn, n=10
    )
)

print("Improved recommendations:")
display(
    recommend_improved(
        most_rated_isbn, n=10
    )
)


In [ ]:

for shrinkage in [10, 20, 50]:

    print(f"\nShrinkage = {shrinkage}")

    display(
        recommend_improved(
            most_rated_isbn,
            n=5,
            shrinkage=shrinkage
        )
    )


In [ ]:

train_popularity = (
    train_data
    .groupby('isbn')
    .agg(
        num_ratings=('rating', 'count'),
        avg_rating=('rating', 'mean')
    )
    .reset_index()
)

C = train_data['rating'].mean()
m = 50

train_popularity['weighted_score'] = (
    train_popularity['num_ratings']
    / (train_popularity['num_ratings'] + m)
    * train_popularity['avg_rating']
    +
    m / (train_popularity['num_ratings'] + m)
    * C
)

popular_isbns = (
    train_popularity
    .sort_values('weighted_score', ascending=False)
    ['isbn']
    .tolist()
)

print("Popularity baseline ready!")


In [ ]:

def recommend_popular_for_user(user_id, n=10):

    seen_books = set(
        train_data.loc[
            train_data['user_id'] == user_id,
            'isbn'
        ]
    )

    recommendations = [
        isbn for isbn in popular_isbns
        if isbn not in seen_books
    ]

    return recommendations[:n]


In [ ]:

hits = 0
evaluated_users = 0

for _, row in test_data.iterrows():

    user_id = row['user_id']
    actual_isbn = row['isbn']

    recommendations = recommend_popular_for_user(
        user_id, n=10
    )

    evaluated_users += 1

    if actual_isbn in recommendations:
        hits += 1

popularity_hit_rate = hits / evaluated_users

print("Evaluated users:", evaluated_users)
print("Hits:", hits)
print(f"Popularity Hit Rate@10: {popularity_hit_rate:.4f}")


In [ ]:

comparison = pd.DataFrame({
    'Model': [
        'Popularity-based',
        'Collaborative Filtering'
    ],
    'Hit Rate@10': [
        popularity_hit_rate,
        hit_rate
    ]
})

display(comparison)


In [ ]:

cf_hits = round(hit_rate * 6217)

fair_cf_hit_rate = cf_hits / len(test_data)

print("Collaborative hits:", cf_hits)
print(f"Collaborative Hit Rate@10: {fair_cf_hit_rate:.4%}")
print(f"Popularity Hit Rate@10: {popularity_hit_rate:.4%}")


In [ ]:

test_data['rating'].value_counts().sort_index()


In [ ]:

positive_test = test_data[
    test_data['rating'] >= 7
].copy()

print("Total test interactions:", len(test_data))
print("Positive test interactions:", len(positive_test))
print(
    "Positive percentage:",
    round(len(positive_test) / len(test_data) * 100, 2)
)


In [ ]:

all_ratings = pd.read_csv(
    '../data/processed/explicit_ratings.csv',
    dtype={'isbn': str}
)

print("Total explicit ratings:", len(all_ratings))


In [ ]:

user_stats = (
    all_ratings
    .groupby('user_id')
    .agg(
        total_ratings=('rating', 'count'),
        positive_ratings=(
            'rating',
            lambda x: (x >= 7).sum()
        )
    )
)

eligible_users = user_stats[
    (user_stats['total_ratings'] >= 5) &
    (user_stats['positive_ratings'] >= 1)
].index

eligible_data = all_ratings[
    all_ratings['user_id'].isin(eligible_users)
].copy()

print("Eligible users:", len(eligible_users))
print("Eligible interactions:", len(eligible_data))


In [ ]:

positive_interactions = eligible_data[
    eligible_data['rating'] >= 7
]

test_v2 = (
    positive_interactions
    .groupby('user_id', group_keys=False)
    .sample(n=1, random_state=42)
)

train_v2 = eligible_data.drop(
    test_v2.index
).copy()

print("Training interactions:", len(train_v2))
print("Testing interactions:", len(test_v2))


In [ ]:

train_book_counts = (
    train_v2
    .groupby('isbn')['user_id']
    .nunique()
)

frequent_train_books = train_book_counts[
    train_book_counts >= 5
].index

train_v2_filtered = train_v2[
    train_v2['isbn'].isin(frequent_train_books)
].copy()

print("Training interactions:", len(train_v2_filtered))
print("Training users:", train_v2_filtered['user_id'].nunique())
print("Training books:", train_v2_filtered['isbn'].nunique())


In [ ]:

known_books = set(train_v2_filtered['isbn'])

test_v2['known_book'] = (
    test_v2['isbn'].isin(known_books)
)

print("Total test users:", len(test_v2))

print(
    "Known-book test users:",
    test_v2['known_book'].sum()
)

print(
    "Cold-start test users:",
    (~test_v2['known_book']).sum()
)

print(
    "Catalog coverage:",
    f"{test_v2['known_book'].mean():.2%}"
)


In [ ]:

remaining_user_counts = (
    train_v2_filtered
    .groupby('user_id')
    .size()
)

print(remaining_user_counts.describe())

print(
    "Users with at least one training rating:",
    len(remaining_user_counts)
)


In [ ]:

import joblib
from pathlib import Path

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(knn_model, model_dir / "knn_model.pkl")
joblib.dump(book_user_matrix, model_dir / "book_user_matrix.pkl")

print("KNN model and matrix saved!")


In [ ]:

joblib.dump(train_books, model_dir / "train_books.pkl")
joblib.dump(book_to_idx, model_dir / "book_to_idx.pkl")
joblib.dump(isbn_to_title, model_dir / "isbn_to_title.pkl")

book_metadata = (
    books[
        ['isbn', 'title', 'author', 'image_m', 'image_l']
    ]
    .drop_duplicates(subset='isbn')
)

book_metadata.to_csv(
    model_dir / "book_metadata.csv",
    index=False
)

print("Mappings and book metadata saved!")


In [ ]:

loaded_model = joblib.load(model_dir / "knn_model.pkl")
loaded_matrix = joblib.load(model_dir / "book_user_matrix.pkl")
loaded_books = joblib.load(model_dir / "train_books.pkl")
loaded_mapping = joblib.load(model_dir / "book_to_idx.pkl")

print("Model loaded successfully!")
print("Matrix shape:", loaded_matrix.shape)
print("Total books:", len(loaded_books))

assert loaded_matrix.shape[0] == len(loaded_books)
assert len(loaded_mapping) == len(loaded_books)

test_isbn = loaded_books[0]

distances, indices = loaded_model.kneighbors(
    loaded_matrix[loaded_mapping[test_isbn]],
    n_neighbors=6
)

print("Test ISBN:", test_isbn)
print("Nearest neighbors:", loaded_books[indices[0]])
